# 🃏 The Mind — LLMs + Aprendizaje por Refuerzo

Simulamos el juego cooperativo **The Mind** con 4 agentes LLM que:
- Se comunican libremente (sin revelar sus cartas)
- Aprenden a coordinarse mediante RL (GRPO)
- Desarrollan lenguaje emergente propio

## Hardware recomendado
| Config | Modelo | Dispositivo |
|--------|--------|-------------|
| GPU 4GB | Qwen2.5-1.5B-Instruct + 4bit | cuda |
| CPU 12GB | Qwen2.5-1.5B-Instruct | cpu |
| GPU 4GB (ligero) | Qwen2.5-0.5B-Instruct | cuda |

In [1]:
# ── Instalación de dependencias ──────────────────────────────────────────────
# Ejecuta esta celda solo la primera vez

# !pip install torch transformers peft accelerate bitsandbytes matplotlib tqdm

# Opcional: Unsloth para entrenar más rápido (GPU con CUDA)
# !pip install "unsloth[cu121-ampere-torch240] @ git+https://github.com/unslothai/unsloth.git"

# trl para utilidades adicionales de RL (opcional)
# !pip install trl

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import os
import torch

# Añade el directorio del proyecto al path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from environment import TheMindEnv
from agents import load_base_model, create_agents
from trainer import GRPOTrainer, TrainerConfig
from utils import setup_logging, TrainingMetrics, LanguageAnalyzer

setup_logging(level='INFO')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


PyTorch: 2.10.0+cu128
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB


In [3]:
# ── Configuración de hardware ─────────────────────────────────────────────────

# Detecta automáticamente el mejor dispositivo disponible
if torch.cuda.is_available():
    DEVICE = 'cuda'
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_4BIT = VRAM_GB < 6  # 4-bit si menos de 6GB VRAM
    print(f'Usando GPU ({VRAM_GB:.1f} GB VRAM). 4-bit: {USE_4BIT}')
else:
    DEVICE = 'cpu'
    USE_4BIT = False
    print('Usando CPU (más lento, pero funciona)')

# ── Selección del modelo ──────────────────────────────────────────────────────
# Opciones (de más ligero a más capaz):
#   'Qwen/Qwen2.5-0.5B-Instruct'  — ~1GB RAM/VRAM (muy rápido)
#   'Qwen/Qwen2.5-1.5B-Instruct'  — ~3GB RAM/VRAM (recomendado)
#   'Qwen/Qwen2.5-3B-Instruct'    — ~6GB RAM (solo si tienes suficiente)
#   'microsoft/Phi-3-mini-4k-instruct' — ~8GB RAM (alternativa)

# MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

# Para 4GB VRAM usa 0.5B:
# MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

# Para llama3.2 3B (si tienes 6GB+ VRAM):
# MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

MODEL_NAME = 'unsloth/Llama-3.2-1B-bnb-4bit'

Usando GPU (4.3 GB VRAM). 4-bit: True


In [4]:
# ── [OPCIONAL] Acelerar con Unsloth ──────────────────────────────────────────
# Descomenta si tienes Unsloth instalado (solo GPU)

USE_UNSLOTH = True

if USE_UNSLOTH and DEVICE == 'cuda':
    from unsloth import FastLanguageModel, get_chat_template
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=USE_4BIT,
    )
    base_model = FastLanguageModel.get_peft_model(
        base_model,
        r=8,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        lora_alpha=32,
        lora_dropout=0.1,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    tokenizer = get_chat_template(
    tokenizer,
    # chat_template = "chatml", 
    chat_template = "llama-3", 
    )
    print('Modelo cargado con Unsloth ✓')
else:
    print('Cargando modelo estándar (sin Unsloth)...')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_132778/2698837045.py:7: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, get_chat_template
2026-04-11 01:12:16,657 [INFO] datasets: TensorFlow version 2.19.1 available.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


2026-04-11 01:12:24,302 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unsloth/Llama-3.2-1B-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 01:12:24,317 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Llama-3.2-1B-bnb-4bit/c19aba40b57098abca2320dbbb84026dcc24cae4/config.json "HTTP/1.1 200 OK"
2026-04-11 01:12:24,467 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unsloth/Llama-3.2-1B-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


2026-04-11 01:12:25,066 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/unslothai/other/revision/main "HTTP/1.1 200 OK"
2026-04-11 01:12:25,235 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unslothai/other/resolve/43d9e0f2f19a5d7836895f648dc0e762816acf77/model.safetensors "HTTP/1.1 302 Found"
2026-04-11 01:12:25,290 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unslothai/other/resolve/43d9e0f2f19a5d7836895f648dc0e762816acf77/.gitattributes "HTTP/1.1 307 Temporary Redirect"
2026-04-11 01:12:25,318 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/other/43d9e0f2f19a5d7836895f648dc0e762816acf77/.gitattributes "HTTP/1.1 200 OK"
2026-04-11 01:12:25,333 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unslothai/other/resolve/43d9e0f2f19a5d7836895f648dc0e762816acf77/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 01:12:25,337 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/unslothai/other/re

Modelo cargado con Unsloth ✓


In [5]:
# ── Carga del modelo base ─────────────────────────────────────────────────────

if not USE_UNSLOTH:
    base_model, tokenizer = load_base_model(
        model_name=MODEL_NAME,
        device='auto' if DEVICE == 'cuda' else 'cpu',
        use_4bit=USE_4BIT,
        use_flash_attention=False,  # ponlo True si tienes GPU Ampere+
    )
    print(f'Modelo {MODEL_NAME} cargado ✓')

print(f'\nParámetros totales: {base_model.num_parameters():,}')


Parámetros totales: 1,237,518,336


In [6]:
# ── Crear agentes ─────────────────────────────────────────────────────────────
# shared_lora=True:  todos aprenden juntos (un solo adaptador)
# shared_lora=False: cada agente tiene su propio adaptador LoRA
#                    (más memoria, pero permite especialización)

NUM_PLAYERS = 4
SHARED_LORA = True  # recomendado para 4GB VRAM
LORA_R = 16          # rango LoRA. Más alto = más parámetros (max 16 para hardware limitado)

agents = create_agents(
    model=base_model,
    tokenizer=tokenizer,
    num_players=NUM_PLAYERS,
    device=DEVICE,
    lora_r=LORA_R,
    shared_lora=SHARED_LORA,
)

print(f'{len(agents)} agentes creados ✓')
print(f'LoRA compartida: {SHARED_LORA}')
agents[0].model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750
4 agentes creados ✓
LoRA compartida: True
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# ── SFT: entrenamiento supervisado previo al RL ───────────────────────────────
from sft_trainer import run_sft, verify_sft_quality

print("Iniciando SFT...")
sft_results = run_sft(
    agents=agents,
    tokenizer=tokenizer,
    dataset_path="sft_dataset.json",
    epochs=3,           # 3-5 épocas suele ser suficiente con 250 ejemplos
    batch_size=2,       # reducir a 1 si hay OOM en GPU 4GB
    lr=2e-4,
    max_length=512,
    save_dir="checkpoints/sft",
    device=DEVICE,
    shared_lora=SHARED_LORA,
)

In [ ]:
# ── Verificar calidad del SFT antes de pasar al RL ────────────────────────────
json_ok_rate = verify_sft_quality(agents, tokenizer, num_samples=5, device=DEVICE)

if json_ok_rate >= 0.8:
    print(f"\n✓ SFT exitoso ({json_ok_rate:.0%} JSON válidos). Listo para RL.")
else:
    print(f"\n⚠ Solo {json_ok_rate:.0%} JSON válidos.")
    print("  Opciones: aumentar epochs, reducir lr, o continuar igualmente.")

In [7]:
# ── Test rápido: una partida sin entrenamiento ─────────────────────────────────
print('Ejecutando partida de prueba...')

env = TheMindEnv(num_players=NUM_PLAYERS)
state = env.reset(level=1)

print(f'Manos iniciales:')
for pid, hand in state.hands.items():
    print(f'  Jugador {pid}: {hand}')

# Probar un turno
obs = env.get_observation(0)
decision = agents[0].generate_action(obs)

print(f'\nDecisión del agente 0:')
print(f'  Mensaje:   {decision["message"]}')
print(f'  Acción:    {decision["action"]}')
print(f'  Razonamiento: {decision["reasoning"][:100]}...')

2026-04-11 01:12:39,733 [INFO] environment: Nueva ronda. Nivel 1. Manos: {0: [26], 1: [24], 2: [20], 3: [12]}


Ejecutando partida de prueba...
Manos iniciales:
  Jugador 0: [26]
  Jugador 1: [24]
  Jugador 2: [20]
  Jugador 3: [12]


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_ut


Decisión del agente 0:
  Mensaje:   En este caso, el número de la carta que necesites es 0, no 1, 2, 3, etc.
  Acción:    play
  Razonamiento: En este caso, el número de la carta que necesites es 0, no 1, 2, 3, etc.
Ese es el número de la cart...


In [ ]:
# ── Configuración del entrenamiento ───────────────────────────────────────────

config = TrainerConfig(
    num_episodes=1000,          # empieza con 200 para ver si funciona
    num_levels=8,              # niveles 1 y 2
    group_size=4,              # 4 juegos por actualización GRPO
    lr=1e-5,                   # LR conservador para LLMs pequeños
    kl_coeff=0.01,
    clip_ratio=0.2,
    entropy_bonus=0.01,
    warmup_episodes=20,        # los primeros 20 ep solo exploración
    max_turns_per_episode=100,
    messages_per_turn=True,
    device=DEVICE,
    accumulate_grad_steps=4,   # simula batch de 4 sin más memoria
    checkpoint_every=25,
)

# Para un entrenamiento más largo y serio:
# config.num_episodes = 1000
# config.num_levels = 3

print('Configuración lista ✓')
print(f'  Episodios: {config.num_episodes}')
print(f'  LR: {config.lr}')
print(f'  Group size (GRPO): {config.group_size}')

Configuración lista ✓
  Episodios: 1000
  LR: 1e-05
  Group size (GRPO): 4


In [9]:
# ── Inicializar entrenador ────────────────────────────────────────────────────

metrics = TrainingMetrics()
lang_analyzer = LanguageAnalyzer()

# Hace la celda autosuficiente si no ejecutaste la celda de test.
if 'env' not in globals() or env is None:
    env = TheMindEnv(num_players=NUM_PLAYERS)

trainer = GRPOTrainer(
    agents=agents,
    env=env,
    config=config,
)

print('Entrenador GRPO listo ✓')

Entrenador GRPO listo ✓


In [ ]:
# ── ENTRENAMIENTO ─────────────────────────────────────────────────────────────
# ¡Esta celda puede tardar bastante!
#   - CPU 12GB Qwen2.5-1.5B: ~30-60 seg/episodio
#   - GPU 4GB Qwen2.5-1.5B:  ~5-15 seg/episodio

print('Iniciando entrenamiento...')
print('(Ctrl+C para detener y ver resultados parciales)')

if USE_UNSLOTH:
    os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

try:
    trainer.train(
        metrics=metrics,
        lang_analyzer=lang_analyzer,
        verbose=True,
    )
except KeyboardInterrupt:
    print('\nEntrenamiento interrumpido por el usuario.')

2026-04-11 01:14:14,208 [INFO] environment: Nueva ronda. Nivel 1. Manos: {0: [43], 1: [42], 2: [14], 3: [10]}


Iniciando entrenamiento...
(Ctrl+C para detener y ver resultados parciales)


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Ep    0 | Nivel 1 | Win: 0.00 | Reward: -11.25 | Errores: 1.0


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Ep   10 | Nivel 1 | Win: 0.00 | Reward: -10.40 | Errores: 1.0


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

NotImplementedError: Unsloth: Logits are empty from 2024.11 onwards. To get raw logits again, please set the environment variable `UNSLOTH_RETURN_LOGITS` to `"1" BEFORE starting to train ie before `trainer.train()`. For example:
```
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
trainer.train()
```
No need to restart your console - just add `os.environ['UNSLOTH_RETURN_LOGITS'] = '1'` before trainer.train() and re-run the cell!

In [13]:
# ── Análisis de resultados ────────────────────────────────────────────────────

metrics.print_summary()
metrics.plot('training_metrics_final2.png')

lang_analyzer.print_report()
lang_analyzer.save_log('language_log2.json')


RESUMEN DE ENTRENAMIENTO
Episodios totales: 80
Win rate (último 50): 0.00%
Reward medio (último 50): -10.658
Errores medio (último 50): 1.00

Lenguaje más reciente (ep 19):
  Palabras más comunes: [('esta', 8), ('vez', 6), ('última', 6), ('estoy', 5), ('no', 5)]
  Longitud media msg:   78.9 chars
Métricas guardadas en training_metrics_final2.png

ANÁLISIS DEL LENGUAJE EMERGENTE
Total de mensajes analizados: 755

Estrategias detectadas:
  duda: 229 ocurrencias
  urgencia: 194 ocurrencias
  señal: 166 ocurrencias
  calma: 143 ocurrencias
  acuerdo: 106 ocurrencias

Evolución del vocabulario:
  Fase inicio: ['j1', 'de', 'estoy', 'la', 'que']
  Fase medio: ['que', 'j0', 'j1', 'de', 'no']
  Fase final: ['la', 'espero', 'j1', 'de', 'mi']
  Fase final: ['no', 'j1', 'espero', 'tengo', 'una']
Log guardado en language_log2.json


In [14]:
# ── Ver ejemplos de mensajes recientes ───────────────────────────────────────
import json

print('Últimos 20 mensajes de los agentes:\n')
for msg in lang_analyzer.message_log[-20:]:
    print(f'  Ep {msg["episode"]:3d} | J{msg["player"]}: {msg["message"]}')

Últimos 20 mensajes de los agentes:

  Ep  19 | J1: No tengo información suficiente para tomar una decisión. No puedo revelar mi número ni realizar oper
  Ep  19 | J1: Esta situación parece ser compleja, así que me permitiré dar algunos pasos internos. Aunque no quier
  Ep  19 | J0: Estoy esperando la siguiente carta para poder comenzar a jugar.
  Ep  19 | J1: No tengo información suficiente sobre mis cartas para tomar una decisión.
  Ep  19 | J2: No tengo información suficiente sobre mis cartas para tomar una decisión.
  Ep  19 | J3: J1: No puedo tomar una decisión hasta saber la posición de todos.
  Ep  19 | J0: J1: No tengo información suficiente sobre mis cartas para tomar una decisión. J2: No tengo informaci
  Ep  19 | J1: J1: No tengo información suficiente sobre mis cartas para tomar una decisión. J2: No tengo informaci
  Ep  19 | J2: J1: No tengo información suficiente sobre mis cartas para tomar una decisión.
  Ep  19 | J3: J1: No tengo información suficiente sobre mis cartas 

In [15]:
# ── Vocabulario emergente por fases ───────────────────────────────────────────

vocab = lang_analyzer.get_vocabulary_evolution(n_bins=3)

fase_names = ['Inicio', 'Medio', 'Final']
for i, (bin_idx, words) in enumerate(vocab.items()):
    print(f'\nFase {fase_names[min(i, 2)]}')
    print('Top 10 palabras:')
    for word, count in words[:10]:
        bar = '█' * min(count, 30)
        print(f'  {word:15s} {bar} ({count})')


Fase Inicio
Top 10 palabras:
  j1              ██████████████████████████████ (117)
  de              ██████████████████████████████ (87)
  estoy           ██████████████████████████████ (82)
  la              ██████████████████████████████ (72)
  que             ██████████████████████████████ (66)
  esta            ██████████████████████████████ (66)
  no              ██████████████████████████████ (63)
  siempre         ██████████████████████████████ (63)
  j0              ██████████████████████████████ (62)
  mi              ██████████████████████████████ (59)

Fase Medio
Top 10 palabras:
  que             ██████████████████████████████ (111)
  j0              ██████████████████████████████ (73)
  j1              ██████████████████████████████ (72)
  de              ██████████████████████████████ (67)
  no              ██████████████████████████████ (62)
  para            ██████████████████████████████ (56)
  mi              ██████████████████████████████ (51)
  estoy           ███

In [17]:
# ── Jugar una partida con los agentes entrenados ──────────────────────────────

print('Partida de demostración con agentes entrenados')
print('='*60)

state = env.reset(level=2)  # nivel 2: 2 cartas por jugador

print('Manos (secretas en el juego real):')
for pid, hand in state.hands.items():
    print(f'  J{pid}: {hand}')
print()

turn = 0
max_turns = 50

while not env.is_done() and turn < max_turns:
    turn += 1
    print(f'--- Turno {turn} | Carta en la mesa: {state.table_top} | Vidas: {state.lives} ---')

    for agent in agents:
        if not state.hands[agent.player_id]:
            continue

        obs = env.get_observation(agent.player_id)
        decision = agent.generate_action(obs)

        if decision['message']:
            env.send_message(agent.player_id, decision['message'])
            print(f'  J{agent.player_id} dice: "{decision["message"]}')

        if decision['action'] == 'play':
            card = agent.get_card_to_play(obs)
            if card:
                result = env.play_card(agent.player_id, card)
                icon = '✓' if result.get('correct') else '✗'
                print(f'  J{agent.player_id} juega {card} {icon}')

        if env.is_done():
            break

print()
if state.won or state.round_over:
    print('🏆 ¡Victoria! Cartas jugadas en orden:', state.played_cards)
else:
    print('💀 Derrota. Cartas jugadas:', state.played_cards)

2026-04-10 23:58:27,508 [INFO] environment: Nueva ronda. Nivel 2. Manos: {0: [8, 23], 1: [5, 39], 2: [11, 47], 3: [28, 31]}


Partida de demostración con agentes entrenados
Manos (secretas en el juego real):
  J0: [8, 23]
  J1: [5, 39]
  J2: [11, 47]
  J3: [28, 31]

--- Turno 1 | Carta en la mesa: 0 | Vidas: 1 ---


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  J0 dice: "No tengo suficientes datos para determinar mi próximo movimiento. Por ahora, seguiré esperando infor


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  J3 dice: "No tengo suficientes datos para determinar mi próximo movimiento. Por ahora, seguiré esperando infor
  J3 juega 28 ✓
--- Turno 4 | Carta en la mesa: 28 | Vidas: 1 ---


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  J0 dice: "espero que la próxima carta sea menor.


2026-04-10 23:59:31,348 [INFO] environment: Error: jugador 1 jugó 5 sobre 28. Vidas: 0


  J1 dice: "J0: Me siento cómodo esperando más información sobre mi próxima carta. Aún así, parece que tendremos
  J1 juega 5 ✗

💀 Derrota. Cartas jugadas: [8, 11, 28, 5]


In [18]:
# ── Guardar modelo final ──────────────────────────────────────────────────────

from utils import save_checkpoint

save_checkpoint(
    agents=agents,
    metrics=metrics,
    episode=config.num_episodes,
    output_dir='checkpoints2',
)
print('Modelo guardado ✓')

Checkpoint guardado en: checkpoints2/episode_1500
  Agente 0: 8.76 MB
  Agente 1: 8.76 MB
  Agente 2: 8.76 MB
  Agente 3: 8.76 MB
Modelo guardado ✓


In [19]:
# ── Reanudar entrenamiento desde un checkpoint ────────────────────────────────

from utils import list_checkpoints, load_checkpoint

# 1. Ver qué checkpoints tienes disponibles
list_checkpoints("checkpoints2")

# 2. Cargar el modelo base y crear los agentes igual que siempre (Ejecutar arriba mejor)
# base_model, tokenizer = load_base_model(model_name=MODEL_NAME, ...)
# agents = create_agents(base_model, tokenizer, ...)

  Episodio   Win rate     Reward    Errores
--------------------------------------------
       250      0.00%       0.00       0.00
      1500      0.00%     -10.66       1.00


[{'episode': 250, 'path': 'checkpoints2/episode_250'},
 {'episode': 1500,
  'path': 'checkpoints2/episode_1500',
  'win_rate': 0.0,
  'avg_reward': -10.658000000000001,
  'mistakes': 1.0}]

In [12]:
# Crear también los optimizadores antes de cargar (para restaurar su estado)
from torch.optim import AdamW
optimizers = [AdamW(a.model.parameters(), lr=1e-5) for a in agents]

# 3. Cargar el checkpoint — devuelve las métricas restauradas
RESUME_FROM_EPISODE = 250
metrics = load_checkpoint(
    agents=agents,
    episode=RESUME_FROM_EPISODE,
    output_dir="checkpoints2",
    optimizers=optimizers,   # opcional, pero recomendado
)

# 4. Crear el trainer pasando los optimizadores ya cargados
trainer = GRPOTrainer(agents=agents, env=env, config=config, optimizers=optimizers)
trainer.episode_count = RESUME_FROM_EPISODE  # que sepa desde dónde va

# 5. Ajustar config para entrenar los episodios restantes
config.num_episodes = 1500  # episodios TOTALES, no adicionales
# El scheduler de niveles se recalculará, así que tenlo en cuenta

# 6. Reanudar
print('Reanudando entrenamiento...')
print('(Ctrl+C para detener y ver resultados parciales)')

try:
    trainer.train(
        metrics=metrics,
        lang_analyzer=lang_analyzer,
        verbose=True,
    )
except KeyboardInterrupt:
    print('\nEntrenamiento interrumpido por el usuario.')

  Agente 0 cargado desde checkpoints2/episode_250/agent_0
  Agente 1 cargado desde checkpoints2/episode_250/agent_1


2026-04-10 18:47:23,374 [INFO] environment: Nueva ronda. Nivel 1. Manos: {0: [26], 1: [24], 2: [20], 3: [12]}


  Agente 2 cargado desde checkpoints2/episode_250/agent_2
  Agente 3 cargado desde checkpoints2/episode_250/agent_3
  AVISO: no se encontró estado del optimizador 0
  AVISO: no se encontró estado del optimizador 1
  AVISO: no se encontró estado del optimizador 2
  AVISO: no se encontró estado del optimizador 3

Checkpoint ep 250 cargado. Listo para reanudar.
Reanudando entrenamiento...
(Ctrl+C para detener y ver resultados parciales)


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Ep    0 | Nivel 1 | Win: 0.00 | Reward: -11.03 | Errores: 1.0


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Ep   10 | Nivel 1 | Win: 0.00 | Reward: -10.63 | Errores: 1.0


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

NotImplementedError: Unsloth: Logits are empty from 2024.11 onwards. To get raw logits again, please set the environment variable `UNSLOTH_RETURN_LOGITS` to `"1" BEFORE starting to train ie before `trainer.train()`. For example:
```
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
trainer.train()
```
No need to restart your console - just add `os.environ['UNSLOTH_RETURN_LOGITS'] = '1'` before trainer.train() and re-run the cell!